# 🎨 Kaggle Dual T4 x2 ComfyUI & DiT Studio (30GB VRAM)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/new)
[![GitHub Store](https://img.shields.io/badge/Notebook_Store-Obsidian_Hub-10b981?style=for-the-badge&logo=github)](https://github.com/SudhirDevOps1/t4-gpu-llm-api)

> **⚡ Architecture Highlights:**
> - 🚀 **Dual GPU Power:** 2x Tesla T4 GPUs (30.6 GB VRAM) for large video & diffusion pipelines.
> - 🛠️ **ComfyUI-Manager:** One-click custom node, LoRA, and checkpoint downloader.
> - 🌐 **Cloudflare Tunnel:** Persistent HTTPS connection without opening ports.
> - 🛡️ **Zero-Leak Security:** Reads secrets from Kaggle secrets client; no keys in commit.


In [ ]:
# 1. Hardware Check & Dynamic Fingerprint
import os, sys, time, uuid, socket, hashlib, torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
raw_fp = f"kaggle-comfy-{socket.gethostname()}-{uuid.uuid4()}-{time.time()}"
INSTANCE_ID = "kaggle-comfy-" + hashlib.sha256(raw_fp.encode()).hexdigest()[:8]
print(f"🆔 Unique Session Fingerprint: {INSTANCE_ID}")
print(f"🔥 Available GPUs: {torch.cuda.device_count()}")


In [ ]:
# 2. Clone ComfyUI, ComfyUI-Manager & Install Cloudflare
!git clone https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git /kaggle/working/ComfyUI/custom_nodes/ComfyUI-Manager
!pip install -q -r /kaggle/working/ComfyUI/requirements.txt
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ ComfyUI dependencies installed on Kaggle!")


In [ ]:
# 3. Launch ComfyUI Server & Expose Tunnel
import subprocess, time, re, os

log_path = "/tmp/comfy_tunnel.log"
if os.path.exists(log_path): os.remove(log_path)

subprocess.Popen([
    "python", "/kaggle/working/ComfyUI/main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--fp16-vae"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)

log_file = open(log_path, "w")
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"], stdout=log_file, stderr=subprocess.STDOUT)

public_url = None
for _ in range(35):
    time.sleep(1)
    if os.path.exists(log_path):
        with open(log_path, "r") as f:
            match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", f.read())
            if match:
                public_url = match.group(0)
                break

if public_url:
    print(f"🎨 PUBLIC COMFYUI URL: {public_url}")
    print(f"🆔 Session Instance ID: {INSTANCE_ID}")
else:
    print("❌ Tunnel initialization failed. Check /tmp/comfy_tunnel.log")
